# Installation

In [ ]:
!pip install keras-tuner

# Load SAR Images

In [1]:
import tensorflow as tf
import rasterio
import numpy as np

def load_sar_tiff_tf(path):
    def _read(p):
        with rasterio.open(p.decode()) as src:
            vv = src.read(1).astype(np.float32)
            vh = src.read(2).astype(np.float32)

        # Fixed-range normalization
        vv = np.clip(vv, -35.0, 5.0)
        vh = np.clip(vh, -40.0, 0.0)

        vv = (vv + 35.0) / 40.0
        vh = (vh + 40.0) / 40.0

        img = np.stack([vv, vh], axis=-1)

        # Resize 2048 → 512
        img = tf.image.resize(img, (512, 512), method="bilinear").numpy()

        return img

    img = tf.numpy_function(_read, [path], tf.float32)
    img.set_shape([512, 512, 2])
    return img

# Augmentation

In [2]:
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.rot90(image, tf.random.uniform([], 0, 4, tf.int32))

    # Mild speckle-like noise
    noise = tf.random.normal(tf.shape(image), stddev=0.02)
    image = tf.clip_by_value(image + noise, 0.0, 1.0)

    return image, label

In [3]:
import os

def get_image_paths_and_labels(images_root):
    oil_dir = os.path.join(images_root, "Oil")
    no_oil_dir = os.path.join(images_root, "No_Oil")

    oil_files = [
        os.path.join(oil_dir, f)
        for f in os.listdir(oil_dir)
        if f.lower().endswith(".tif")
    ]

    no_oil_files = [
        os.path.join(no_oil_dir, f)
        for f in os.listdir(no_oil_dir)
        if f.lower().endswith(".tif")
    ]

    paths = oil_files + no_oil_files
    labels = [1] * len(oil_files) + [0] * len(no_oil_files)

    return paths, labels

# Dataset Preparation

In [4]:
BATCH_SIZE = 8

def make_dataset(paths, labels, shuffle=False, augment_data=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), reshuffle_each_iteration=True)

    ds = ds.map(
        lambda x, y: (load_sar_tiff_tf(x), y),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if augment_data:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

In [6]:
DATASET_ROOT = "/Volumes/Windows8_OS/Dataset/Dataset-OG" # Change Path

TRAIN_DIR = os.path.join(DATASET_ROOT, "Train", "images") # Confirm Folder Name
TEST_DIR  = os.path.join(DATASET_ROOT, "Test", "images") # Confirm Folder Name

train_paths, train_labels = get_image_paths_and_labels(TRAIN_DIR)
test_paths,  test_labels  = get_image_paths_and_labels(TEST_DIR)

print("Train:", len(train_paths))
print("Test:", len(test_paths))

train_ds = make_dataset(
    train_paths, train_labels,
    shuffle=True,
    augment_data=True
)

test_ds = make_dataset(
    test_paths, test_labels,
    shuffle=False,
    augment_data=False
)

Train: 1885
Test: 300


I0000 00:00:1768406163.010704 1801603 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1768406163.011890 1801603 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


# Model

In [7]:
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def build_detection_cnn(hp):
    model = Sequential()

    # Hyperparameter: number of conv blocks
    num_blocks = hp.Int("num_conv_blocks", min_value=3, max_value=8)

    for i in range(num_blocks):
        filters = hp.Choice(f"filters_{i}", [16, 32, 64])

        if i == 0:
            model.add(Conv2D(
                filters,
                (3, 3),
                activation="relu",
                padding="same",
                input_shape=(512, 512, 2)
            ))
        else:
            model.add(Conv2D(
                filters,
                (3, 3),
                activation="relu",
                padding="same"
            ))

        model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Flatten())

    # Hyperparameter: dense units
    dense_units = hp.Choice("dense_units", [16, 32, 64])
    model.add(Dense(dense_units, activation="relu"))

    # Hyperparameter: dropout
    model.add(Dropout(
        hp.Float("dropout", min_value=0.2, max_value=0.5, step=0.1)
    ))

    model.add(Dense(1, activation="sigmoid"))

    # Hyperparameter: learning rate
    lr = hp.Choice("learning_rate", [1e-3, 1e-4, 3e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

# Fine Tuner

In [8]:
tuner = kt.RandomSearch(
    build_detection_cnn,
    objective="val_accuracy",
    max_trials=20,
    executions_per_trial=1,
    directory="cnn_tuning_logs_big_dataset",
    project_name="sar_oil_detection"
)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [12]:
tuner.search(
    train_ds,
    validation_data=test_ds,
    epochs=50,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Trial 1 Complete [08h 04m 06s]
val_accuracy: 0.6233333349227905

Best val_accuracy So Far: 0.6233333349227905
Total elapsed time: 08h 04m 06s

Search: Running Trial #2

Value             |Best Value So Far |Hyperparameter
4                 |8                 |num_conv_blocks
64                |16                |filters_0
32                |64                |filters_1
16                |64                |filters_2
16                |16                |dense_units
0.4               |0.2               |dropout
0.0001            |0.0001            |learning_rate
64                |16                |filters_3
32                |16                |filters_4
32                |16                |filters_5
64                |16                |filters_6
32                |16                |filters_7

Epoch 1/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 2636s 11s/step - accuracy: 0.6700 - loss: 0.6257 - val_accuracy: 0.6300 - val_loss: 0.6079 - learning_rate: 1.0000e-04
Epoch 2/50
236/236 ━━━━━━━━━━━━━

KeyboardInterrupt: 

In [ ]:
best_model = tuner.get_best_models(1)[0]

# Model Fitting

In [ ]:
history = best_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=50,
    callbacks=callbacks
)

In [ ]:
train_loss, train_acc = best_model.evaluate(train_ds)
print(f"Train Acc -> {train_acc}, Train Loss -> {train_loss}")

In [ ]:
validate_loss, validate_acc = best_model.evaluate(val_ds)
print(f"Validation Acc -> {validate_acc}, Validation Loss -> {validate_loss}")

In [ ]:
test_loss, test_acc = best_model.evaluate(test_ds)
print(f"Test Acc -> {test_acc}, Test Loss -> {test_loss}")

# Plots

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history.history["accuracy"]) + 1)

plt.figure(figsize=(10, 6))
plt.plot(epochs, history.history["accuracy"], label="Training Accuracy", linewidth=2)
plt.plot(epochs, history.history["val_accuracy"], label="Validation Accuracy", linewidth=2)

plt.title("Training vs Validation Accuracy", fontsize=16)
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Accuracy", fontsize=12)

plt.legend(fontsize=11)
plt.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(epochs, history.history["loss"], label="Training Loss", linewidth=2)
plt.plot(epochs, history.history["val_loss"], label="Validation Loss", linewidth=2)

plt.title("Training vs Validation Loss", fontsize=16)
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Loss", fontsize=12)

plt.legend(fontsize=11)
plt.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
best_epoch = history.history["val_accuracy"].index(
    max(history.history["val_accuracy"])
) + 1

plt.axvline(best_epoch, linestyle=":", linewidth=2)

In [ ]:
plt.savefig("accuracy_curve.png", dpi=300, bbox_inches="tight")
plt.savefig("loss_curve.png", dpi=300, bbox_inches="tight")

# Prediction Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_test_predictions(model, dataset, class_names=("No_Oil", "Oil"), num_images=24, cols=4):

    images, labels = next(iter(dataset))

    # Run prediction
    preds = model.predict(images)
    preds = preds.squeeze()

    num_images = min(num_images, images.shape[0])
    rows = int(np.ceil(num_images / cols))

    plt.figure(figsize=(cols * 4, rows * 4))

    for i in range(num_images):
        plt.subplot(rows, cols, i + 1)

        # Show VV (grayscale) channel
        img = images[i, :, :, 0]
        plt.imshow(img, cmap="gray")
        plt.axis("off")

        true_label = labels[i].numpy()
        pred_prob = preds[i]

        plt.title(
            f"True label = {true_label:.2f}\nPrediction = {pred_prob:.3g}",
            fontsize=10
        )

    plt.tight_layout()
    plt.show()

In [ ]:
plot_test_predictions(
    model=best_model,
    dataset=test_ds,
    num_images=28,
    cols=4
)

# Model Save

In [ ]:
model.export("/Users/parasningune/Desktop/Best_Models/512_90.42_91.67/model") # Change Path

In [ ]:
model.save("/Users/parasningune/Desktop/Best_Models/512_90.42_91.67/model.h5") # Change Path

In [ ]:
model.save("/Users/parasningune/Desktop/Best_Models/512_90.42_91.67/model.keras") # Change Path

In [ ]:
model.save_weights("/Users/parasningune/Desktop/Best_Models/512_90.42_91.67/model.weights.h5") # Change Path